# rp1 Robustness: the leverage cap k

**Question: do the portfolio results depend on the leverage cap?** The main portfolios cap the base
weight at k = 2,

wᵢ,ₜ = min(σ̄²ᵢ / σ̂²ᵢ,ₜ₊₁, k).

This notebook rebuilds every model's cyclical bucket with **k = 1.5**, **k = 3** and **no cap**
(k = ∞), next to the main k = 2, and checks whether the Sharpe ratio and alpha results hold.
Everything else is as in `p1_build_portfolios.ipynb`: the same forecasts, the fixed σ̄² (the main
specification), the 1/N bucket and the rescale to buy-and-hold volatility.

**What the cap changes.** Every portfolio is still rescaled by c to the volatility of buy-and-hold,
so the cap does not change the average leverage. It changes the shape of the weights: a low cap
clips the large bets in calm months, and c then levers the whole portfolio back up; no cap lets
the calm-month bets grow as large as the forecast implies.

**Measures**, the same as `p2_portfolio_eval.ipynb`, for the cyclical bucket: the annualised Sharpe
ratio with the Ledoit-Wolf test against buy-and-hold, and the alpha on buy-and-hold (Newey-West).

Run `p1_build_portfolios.ipynb` first: this notebook reads its out-of-sample window and checks
that its k = 2 rebuild matches p1's saved portfolios.

## Setup

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

_checkout = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "models" / "research_config.py").is_file()),
    None,
)
if _checkout is None:
    raise FileNotFoundError("Start the notebook kernel from within this project checkout.")
sys.path.insert(0, str(_checkout / "models"))   # models/: research_config, portfolio.config and utils


from research_config import SELECTED, TICKERS, TRAIN_START, TRAIN_MONTHS
from portfolio.config import FORECAST_COL, K_CAP
from utils.portfolio_utils import (
    PORTFOLIO_DIR,
    load_monthly_returns, training_window, load_forecast_files, forecast_wide, load_portfolios,
    build_portfolio, sigma2_bar_fixed, evaluate_bucket, pct_capped_pooled,
)

CAPS = {"k = 1.5": 1.5, "k = 2 (main)": 2.0, "k = 3": 3.0, "no cap": np.inf}
MAIN = "k = 2 (main)"
assert CAPS[MAIN] == K_CAP, "the main cap in CAPS must match K_CAP in portfolio/config.py"


OUT_DIR = _checkout / "models" / "portfolio" / "robustness" / "results"
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.3f}".format)

## Rebuild the portfolios under each cap

The forecasts, out-of-sample window and fixed σ̄² are those of p1. The k = 2 rebuild must reproduce
p1's saved portfolios exactly, which confirms the other caps differ from the main results only in
the cap.

In [2]:
run_info = json.loads((PORTFOLIO_DIR / "run_info.json").read_text())
forecasts_raw, available = load_forecast_files(SELECTED)
MODELS = [v for variants in available.values() for v in variants]
assert MODELS == run_info["variants"], "the models differ from the last run of p1; rerun p1 first"

saved, xret = load_portfolios("fixed")   # p1's portfolios (k = 2) and the out-of-sample returns
sigma2_hat = {v: forecast_wide(forecasts_raw[f], v, FORECAST_COL, TICKERS).loc[xret.index]
              for f, variants in available.items() for v in variants}
_, rv_monthly = load_monthly_returns(TICKERS)
train_idx, _ = training_window(TRAIN_START, TRAIN_MONTHS)
sigma2_bar = sigma2_bar_fixed(rv_monthly[TICKERS], train_idx)
bh = xret.mean(axis=1)

portfolios = {cap: {m: build_portfolio(sigma2_hat[m], xret, sigma2_bar, k) for m in MODELS}
              for cap, k in CAPS.items()}

for m in MODELS:
    assert np.allclose(portfolios[MAIN][m]["managed"], saved[m]["managed"]), f"{m}: k = 2 rebuild differs from p1"
print(f"{len(MODELS)} models x {len(CAPS)} caps, {xret.index.min():%Y-%m} to {xret.index.max():%Y-%m} "
      f"({len(xret)} months). The k = 2 rebuild matches p1.")

11 models x 4 caps, 2008-03 to 2025-12 (214 months). The k = 2 rebuild matches p1.


## Sharpe ratio and alpha under each cap. Model against Buy and Hold benchmark

One row per cap and model. `p vs BH` is the Ledoit-Wolf p-value of SR(model) = SR(BH); `α` is the
alpha on buy-and-hold with Newey-West errors, annualised. `% capped` is the share of ETF-months at
the cap and `max weight` the largest base weight wᵢ,ₜ: together they show how hard each cap binds.
(About 20 seconds: one Ledoit-Wolf bootstrap per row.)

In [3]:
rows = {}
for cap, res in portfolios.items():
    for m in MODELS:
        ev = evaluate_bucket(res[m], bh)
        rows[(cap, m)] = {
            "SR": ev["ann_sharpe"],
            "ΔSR vs BH": ev["SR_diff_vs_BH"],
            "p vs BH": ev["LW_p"],
            "α % p.a.": ev["alpha_annual"] * 100,
            "α t": ev["alpha_t"],
            "α p": ev["alpha_p"],
            "% capped": pct_capped_pooled(res[m]["weights_base"], CAPS[cap]),
            "max weight": res[m]["weights_base"].to_numpy().max(),
            "c": res[m]["c"],
        }
results = pd.DataFrame(rows).T.astype(float).rename_axis(["cap", "model"])

sr_bh = bh.mean() / bh.std() * np.sqrt(12)
print(f"Buy-and-hold Sharpe ratio: {sr_bh:.3f}")
results

Buy-and-hold Sharpe ratio: 0.476


SR  ΔSR vs BH  p vs BH  α % p.a.    α t   α p  % capped  max weight     c
cap          model                                                                                          
k = 1.5      hist_rw            0.521      0.045    0.823     2.877  0.908 0.364    34.393       1.500 1.336
             garch              0.509      0.033    0.856     2.521  0.838 0.402    16.262       1.500 1.455
             garch_midas_baa10y 0.510      0.034    0.854     2.517  0.835 0.403    16.636       1.500 1.450
             en_no_macro        0.468     -0.008    0.967     1.675  0.545 0.586    14.579       1.500 1.554
             en_no_vix          0.425     -0.051    0.776     0.522  0.177 0.860     9.439       1.500 1.564
             en_vix_only        0.455     -0.021    0.913     1.422  0.467 0.641     9.813       1.500 1.566
             en_full_macro      0.420     -0.056    0.751     0.594  0.204 0.838     8.411       1.500 1.615
             xgb_no_macro       0.474     -0.002    0.991     1.548  0.580 0.562     5.607       1.500 1.650
             xgb_no_vix         0.475     -0.001    0.996     1.740  0.574 0.566     6.916       1.500 1.653
             xgb_vix_only       0.407     -0.070    0.694     0.498  0.169 0.866     7.383       1.500 1.673
             xgb_full_macro     0.373     -0.103    0.569    -0.045 -0.015 0.988    10.000       1.500 1.676
k = 2 (main) hist_rw            0.468     -0.008    0.969     2.175  0.644 0.520    21.308       2.000 1.175
             garch              0.497      0.021    0.912     2.451  0.787 0.431     4.393       2.000 1.381
             garch_midas_baa10y 0.499      0.023    0.903     2.465  0.787 0.431     4.299       2.000 1.374
             en_no_macro        0.453     -0.023    0.911     1.610  0.499 0.617     5.327       2.000 1.454
             en_no_vix          0.428     -0.048    0.798     0.743  0.240 0.810     2.804       2.000 1.501
             en_vix_only        0.459     -0.017    0.927     1.637  0.518 0.604     2.243       2.000 1.507
             en_full_macro      0.428     -0.048    0.795     0.834  0.277 0.782     1.028       2.000 1.576
             xgb_no_macro       0.467     -0.009    0.952     1.504  0.537 0.591     2.430       2.000 1.604
             xgb_no_vix         0.477      0.001    0.994     1.873  0.601 0.548     1.215       2.000 1.611
             xgb_vix_only       0.415     -0.061    0.737     0.748  0.244 0.807     1.869       2.000 1.632
             xgb_full_macro     0.370     -0.106    0.578     0.058  0.018 0.985     2.710       2.000 1.608
k = 3        hist_rw            0.417     -0.059    0.796     1.655  0.456 0.648     8.318       3.000 1.004
             garch              0.504      0.029    0.884     2.675  0.844 0.399     0.093       3.000 1.354
             garch_midas_baa10y 0.509      0.033    0.862     2.732  0.860 0.390     0.093       3.000 1.351
             en_no_macro        0.466     -0.010    0.964     1.996  0.603 0.546     0.561       3.000 1.405
             en_no_vix          0.441     -0.036    0.850     1.022  0.326 0.745     0.187       3.000 1.487
             en_vix_only        0.462     -0.014    0.938     1.731  0.541 0.588     0.187       3.000 1.491
             en_full_macro      0.432     -0.044    0.817     0.934  0.309 0.758     0.093       3.000 1.572
             xgb_no_macro       0.471     -0.005    0.972     1.615  0.575 0.566     0.000       2.799 1.588
             xgb_no_vix         0.481      0.006    0.974     1.973  0.631 0.528     0.000       2.314 1.606
             xgb_vix_only       0.426     -0.050    0.783     1.008  0.323 0.747     0.000       2.705 1.615
             xgb_full_macro     0.374     -0.102    0.596     0.175  0.054 0.957     0.093       3.000 1.588
no cap       hist_rw            0.456     -0.020    0.931     2.766  0.725 0.469     0.000      13.976 0.904
             garch              0.505      0.029    0.881     2.694  0.849 0.396     0.000       3.824 1.354
  

## Against buy-and-hold, by cap

The cyclical bucket of each model (rows) under each cap (column groups): ΔSR = SR(model) − SR(BH)
with its Ledoit-Wolf p-value, and the alpha on buy-and-hold (% p.a.) with its Newey-West p-value.

In [4]:
COLS = ["ΔSR vs BH", "p vs BH", "α % p.a.", "α p"]
by_cap = (results[COLS].unstack("cap")
          .swaplevel(axis=1)
          .reindex(columns=pd.MultiIndex.from_product([list(CAPS), COLS]))
          .loc[MODELS])
print(f"Buy-and-hold Sharpe ratio: {sr_bh:.3f}")
by_cap

Buy-and-hold Sharpe ratio: 0.476


k = 1.5                        k = 2 (main)                            k = 3                           no cap                       
                   ΔSR vs BH p vs BH α % p.a.   α p    ΔSR vs BH p vs BH α % p.a.   α p ΔSR vs BH p vs BH α % p.a.   α p ΔSR vs BH p vs BH α % p.a.   α p
model                                                                                                                                                    
hist_rw                0.045   0.823    2.877 0.364       -0.008   0.969    2.175 0.520    -0.059   0.796    1.655 0.648    -0.020   0.931    2.766 0.469
garch                  0.033   0.856    2.521 0.402        0.021   0.912    2.451 0.431     0.029   0.884    2.675 0.399     0.029   0.881    2.694 0.396
garch_midas_baa10y     0.034   0.854    2.517 0.403        0.023   0.903    2.465 0.431     0.033   0.862    2.732 0.390     0.034   0.860    2.744 0.388
en_no_macro           -0.008   0.967    1.675 0.586       -0.023   0.911    1.610 0.617    -0.010   0.964    1.996 0.546    -0.001   0.996    2.183 0.512
en_no_vix             -0.051   0.776    0.522 0.860       -0.048   0.798    0.743 0.810    -0.036   0.850    1.022 0.745    -0.034   0.862    1.062 0.735
en_vix_only           -0.021   0.913    1.422 0.641       -0.017   0.927    1.637 0.604    -0.014   0.938    1.731 0.588    -0.011   0.952    1.799 0.575
en_full_macro         -0.056   0.751    0.594 0.838       -0.048   0.795    0.834 0.782    -0.044   0.817    0.934 0.758    -0.043   0.822    0.949 0.754
xgb_no_macro          -0.002   0.991    1.548 0.562       -0.009   0.952    1.504 0.591    -0.005   0.972    1.615 0.566    -0.005   0.972    1.615 0.566
xgb_no_vix            -0.001   0.996    1.740 0.566        0.001   0.994    1.873 0.548     0.006   0.974    1.973 0.528     0.006   0.974    1.973 0.528
xgb_vix_only          -0.070   0.694    0.498 0.866       -0.061   0.737    0.748 0.807    -0.050   0.783    1.008 0.747    -0.050   0.783    1.008 0.747
xgb_full_macro        -0.103   0.569   -0.045 0.988       -0.106   0.578    0.058 0.985    -0.102   0.596    0.175 0.957    -0.102   0.598    0.186 0.954

## Save

`results/rp1_different_leverages.csv`: the table above, one row per cap and model.

In [5]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(OUT_DIR / "rp1_different_leverages.csv")
print("Saved to", (OUT_DIR / "rp1_different_leverages.csv").resolve())

Saved to C:\Documents\NUS\Y4S1\DSE4101\dse4101capstone_project\models\portfolio\robustness\results\rp1_different_leverages.csv
